# Notebook 02: Dashboards de BI em Plotly

**Projeto:** Predição de Avaliações Negativas no Ecossistema Olist<br>
**Autor:** Rafael de Menezes Ehlers<br>
**Fase:** 2 (Implementação)<br>
**Curso:** Curso Superior de Tecnologia em Banco de Dados

## Objetivo

Este notebook consome o Data Warehouse `olist_dw.sqlite` (gerado pelo Notebook 01) e produz **quatro dashboards interativos em Plotly**, cada um respondendo a uma pergunta específica de negócio:

| Dashboard | Pergunta | Hipótese testada |
|---|---|---|
| 0. Visão Executiva | Qual a saúde geral do negócio Olist? | Contexto |
| 1. Atrasos e Satisfação | Atrasos geram avaliações negativas? | **H1** |
| 2. Frete vs Preço em Baixo Ticket | Frete pesa mais que preço em produtos baratos? | **H2** |
| 3. Pedidos Multi-Vendedor | Multi-sellers geram mais avaliações negativas? | **H3** |

## Saída

Quatro arquivos HTML standalone serão criados em `/content/drive/MyDrive/PUCRS/projetobi-olist/dashboards/`



## 1. Setup do ambiente

In [61]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [62]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Caminhos
BASE_PATH = '/content/drive/MyDrive/PUCRS/projetobi-olist'
DW_PATH = os.path.join(BASE_PATH, 'olist_dw.sqlite')
DASHBOARDS_PATH = os.path.join(BASE_PATH, 'dashboards')
os.makedirs(DASHBOARDS_PATH, exist_ok=True)

# Conexao com o Data Warehouse
engine = create_engine(f'sqlite:///{DW_PATH}')

print(f'Data Warehouse: {DW_PATH}')
print(f'Pasta de saida: {DASHBOARDS_PATH}')

# Quick check
n = pd.read_sql('SELECT COUNT(*) AS n FROM fato_pedidos', engine).iloc[0, 0]
print(f'Pedidos na fato: {n:,}')

Data Warehouse: /content/drive/MyDrive/PUCRS/projetobi-olist/olist_dw.sqlite
Pasta de saida: /content/drive/MyDrive/PUCRS/projetobi-olist/dashboards
Pedidos na fato: 95,824


### Função auxiliar para salvar dashboards

Combina múltiplas figuras Plotly em um único arquivo HTML standalone, com título, descrição contextual e Plotly.js carregado uma única vez via CDN.

In [63]:
def save_dashboard(figs, descriptions, title, subtitle, output_path):
    """
    Combina figuras Plotly em um unico HTML standalone.

    figs: lista de figuras Plotly
    descriptions: lista de strings descritivas, uma por figura
    title: titulo principal do dashboard
    subtitle: subtitulo explicativo
    output_path: caminho de saida do arquivo HTML
    """
    blocks = []
    for fig, desc in zip(figs, descriptions):
        blocks.append(f'<div class="section"><p class="desc">{desc}</p>')
        blocks.append(fig.to_html(include_plotlyjs=False, full_html=False))
        blocks.append('</div>')

    html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<title>{title}</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
       max-width: 1400px; margin: 0 auto; padding: 24px; color: #1f2937; background: #f3f4f6; }}
header {{ background: white; padding: 24px; border-radius: 12px; margin-bottom: 24px; border-left: 5px solid #3b82f6; }}
h1 {{ color: #111827; margin: 0 0 8px 0; font-size: 24px; }}
.subtitle {{ color: #6b7280; margin: 0; font-size: 14px; }}
.section {{ background: white; padding: 20px; border-radius: 12px; margin-bottom: 16px; }}
.desc {{ color: #4b5563; font-size: 14px; margin: 0 0 16px 0; font-style: italic; }}
</style>
</head>
<body>
<header>
<h1>{title}</h1>
<p class="subtitle">{subtitle}</p>
</header>
{''.join(blocks)}
</body>
</html>"""

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f'Dashboard salvo: {output_path}')

## 2. Dashboard 0: Visão Executiva

KPIs estratégicos e visão macro do ecossistema Olist no período 2016 a 2018.

In [64]:
# Query unica para todos os KPIs principais
kpis = pd.read_sql('''
SELECT
    SUM(valor_total_pedido) AS faturamento_total,
    COUNT(*) AS total_pedidos,
    AVG(valor_total_pedido) AS ticket_medio,
    AVG(tempo_total_entrega_dias) AS prazo_medio,
    100.0 * SUM(atraso) / COUNT(*) AS taxa_atraso,
    100.0 * SUM(avaliacao_negativa) / COUNT(*) AS taxa_neg,
    100.0 * SUM(multi_vendedor) / COUNT(*) AS taxa_multi,
    100.0 * SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) / COUNT(*) -
    100.0 * SUM(CASE WHEN review_score <= 3 THEN 1 ELSE 0 END) / COUNT(*) AS nps_estimado
FROM fato_pedidos
''', engine).iloc[0]

# Grid 2x4 de KPI cards
fig_d0_1 = make_subplots(
    rows=2, cols=4,
    specs=[[{'type': 'indicator'}]*4]*2,
    subplot_titles=('Faturamento Total', 'Pedidos', 'Ticket Médio', 'NPS Estimado',
                    'Prazo Medio Entrega', 'Taxa de Atraso', 'Taxa Aval. Negativa', 'Multi-Vendedor')
)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['faturamento_total']),
                                 number={'prefix': 'R$ ', 'valueformat': ',.0f'}), row=1, col=1)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['total_pedidos']),
                                 number={'valueformat': ',.0f'}), row=1, col=2)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['ticket_medio']),
                                 number={'prefix': 'R$ ', 'valueformat': ',.2f'}), row=1, col=3)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['nps_estimado']),
                                 number={'suffix': '%', 'valueformat': '.1f'}), row=1, col=4)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['prazo_medio']),
                                 number={'suffix': ' dias', 'valueformat': '.1f'}), row=2, col=1)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['taxa_atraso']),
                                 number={'suffix': '%', 'valueformat': '.2f'}), row=2, col=2)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['taxa_neg']),
                                 number={'suffix': '%', 'valueformat': '.2f'}), row=2, col=3)
fig_d0_1.add_trace(go.Indicator(mode='number', value=float(kpis['taxa_multi']),
                                 number={'suffix': '%', 'valueformat': '.2f'}), row=2, col=4)
fig_d0_1.update_layout(height=400, margin=dict(t=60, b=20))
fig_d0_1

In [65]:
# Faturamento e pedidos mensais
mensal = pd.read_sql('''
SELECT t.ano, t.mes,
       SUM(f.valor_total_pedido) AS faturamento,
       COUNT(*) AS pedidos
FROM fato_pedidos f
JOIN dim_tempo t ON f.tempo_key = t.data_key
GROUP BY t.ano, t.mes
ORDER BY t.ano, t.mes
''', engine)
mensal['data'] = pd.to_datetime(dict(year=mensal['ano'], month=mensal['mes'], day=1))

fig_d0_2 = make_subplots(specs=[[{'secondary_y': True}]])
fig_d0_2.add_trace(go.Bar(x=mensal['data'], y=mensal['faturamento'],
                          name='Faturamento (R$)', marker_color='#3b82f6'), secondary_y=False)
fig_d0_2.add_trace(go.Scatter(x=mensal['data'], y=mensal['pedidos'],
                              name='Pedidos', mode='lines+markers', line=dict(color='#ef4444', width=2)),
                   secondary_y=True)
fig_d0_2.update_layout(title='Faturamento e volume de pedidos por mês (2016-2018)',
                       height=400, hovermode='x unified')
fig_d0_2.update_yaxes(title_text='Faturamento (R$)', secondary_y=False)
fig_d0_2.update_yaxes(title_text='Pedidos', secondary_y=True)
fig_d0_2

In [66]:
# Top estados por faturamento
estados = pd.read_sql('''
SELECT c.estado_cliente,
       SUM(f.valor_total_pedido) AS faturamento,
       COUNT(*) AS pedidos
FROM fato_pedidos f
JOIN dim_cliente c ON f.cliente_key = c.customer_id
GROUP BY c.estado_cliente
ORDER BY faturamento DESC
''', engine)

fig_d0_3 = px.bar(
    estados.head(15).sort_values('faturamento'),
    x='faturamento', y='estado_cliente', orientation='h',
    title='Top 15 estados por faturamento',
    labels={'faturamento': 'Faturamento (R$)', 'estado_cliente': 'Estado'},
    color='faturamento', color_continuous_scale='Blues'
)
fig_d0_3.update_layout(height=500, showlegend=False)
fig_d0_3

In [67]:
# Distribuicao de notas
notas = pd.read_sql('''
SELECT review_score, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fato_pedidos), 2) AS pct
FROM fato_pedidos GROUP BY review_score ORDER BY review_score
''', engine)

fig_d0_4 = px.bar(
    notas, x='review_score', y='qtd', text='pct',
    title='Distribuição de notas de review',
    labels={'review_score': 'Nota (estrelas)', 'qtd': 'Quantidade de pedidos'},
    color='review_score', color_continuous_scale='RdYlGn'
)
fig_d0_4.update_traces(texttemplate='%{text}%', textposition='outside')
fig_d0_4.update_layout(height=400, showlegend=False, coloraxis_showscale=False)
fig_d0_4

In [68]:
# Salvar Dashboard 0
save_dashboard(
    figs=[fig_d0_1, fig_d0_2, fig_d0_3, fig_d0_4],
    descriptions=[
        'Indicadores estrategicos do ecossistema Olist no periodo 2016 a 2018.',
        'Evolução do faturamento (barras, eixo esquerdo) e do volume de pedidos (linha, eixo direito).',
        'Concentração geográfica das vendas: 15 estados que mais contribuem para o faturamento total.',
        'Distribuição das notas atribuídas pelos clientes (1 a 5 estrelas).'
    ],
    title='Dashboard 0: Visão Executiva',
    subtitle='KPIs estratégicos e visão macro do ecossistema Olist',
    output_path=os.path.join(DASHBOARDS_PATH, 'dashboard_0_visao_executiva.html')
)

Dashboard salvo: /content/drive/MyDrive/PUCRS/projetobi-olist/dashboards/dashboard_0_visao_executiva.html


## 3. Dashboard 1: Hipótese 1 — Atrasos e Satisfação

**Hipótese 1:** pedidos cujo tempo real de entrega ultrapassa o prazo estimado apresentam probabilidade significativamente maior de receberem avaliação negativa.

In [69]:
# Histograma do delta de entrega
df_delta = pd.read_sql('''
SELECT delta_entrega_dias FROM fato_pedidos
WHERE delta_entrega_dias BETWEEN -30 AND 30
''', engine)

fig_d1_1 = px.histogram(
    df_delta, x='delta_entrega_dias', nbins=60,
    title='Distribuição do delta de entrega',
    labels={'delta_entrega_dias': 'Delta (entrega real - estimada, em dias)', 'count': 'Pedidos'}
)
fig_d1_1.add_vline(x=0, line_dash='dash', line_color='red',
                    annotation_text='Limite do prazo', annotation_position='top')
fig_d1_1.update_layout(height=400)
fig_d1_1

In [70]:
# PUNCH LINE da Hipotese 1
h1_taxa = pd.read_sql('''
SELECT CASE WHEN atraso = 1 THEN 'Atrasado' ELSE 'No prazo' END AS situacao,
       100.0 * AVG(avaliacao_negativa) AS taxa_neg_pct,
       COUNT(*) AS n_pedidos
FROM fato_pedidos GROUP BY atraso
''', engine)

fig_d1_2 = px.bar(
    h1_taxa, x='situacao', y='taxa_neg_pct', text='taxa_neg_pct',
    title='HIPÓTESE 1 - Taxa de avaliação negativa: no prazo vs atrasado',
    labels={'situacao': 'Status do prazo', 'taxa_neg_pct': 'Taxa de aval. negativa (%)'},
    color='situacao',
    color_discrete_map={'Atrasado': '#dc2626', 'No prazo': '#16a34a'}
)
fig_d1_2.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_d1_2.update_layout(height=420, showlegend=False)
fig_d1_2

In [71]:
# Box plot delta x review_score
df_box = pd.read_sql('''
SELECT review_score, delta_entrega_dias FROM fato_pedidos
WHERE delta_entrega_dias BETWEEN -60 AND 60
''', engine)

# Cast para string para que o Plotly trate como categorico
df_box['review_score'] = df_box['review_score'].astype(str)

fig_d1_3 = px.box(
    df_box, x='review_score', y='delta_entrega_dias',
    title='Delta de entrega por nota de review',
    labels={'review_score': 'Nota de review', 'delta_entrega_dias': 'Delta (dias)'},
    color='review_score',
    color_discrete_map={'1': '#dc2626', '2': '#f97316', '3': '#eab308', '4': '#84cc16', '5': '#16a34a'},
    category_orders={'review_score': ['1', '2', '3', '4', '5']}
)
fig_d1_3.add_hline(y=0, line_dash='dash', line_color='gray')
fig_d1_3.update_layout(height=400, showlegend=False)
fig_d1_3

In [72]:
# Taxa neg por faixa de atraso
faixa_atraso = pd.read_sql('''
SELECT
    CASE
        WHEN delta_entrega_dias <= -5 THEN '1. Adiantado >5d'
        WHEN delta_entrega_dias < 0 THEN '2. Adiantado <5d'
        WHEN delta_entrega_dias = 0 THEN '3. No dia'
        WHEN delta_entrega_dias <= 3 THEN '4. Atraso ate 3d'
        WHEN delta_entrega_dias <= 7 THEN '5. Atraso 4-7d'
        WHEN delta_entrega_dias <= 15 THEN '6. Atraso 8-15d'
        ELSE '7. Atraso >15d'
    END AS faixa,
    100.0 * AVG(avaliacao_negativa) AS taxa_neg_pct,
    COUNT(*) AS n_pedidos
FROM fato_pedidos
GROUP BY faixa
ORDER BY faixa
''', engine)

fig_d1_4 = px.line(
    faixa_atraso, x='faixa', y='taxa_neg_pct', markers=True,
    title='Taxa de avaliação negativa cresce com o tamanho do atraso',
    labels={'faixa': 'Faixa de atraso', 'taxa_neg_pct': 'Taxa de aval. negativa (%)'}
)
fig_d1_4.update_traces(line=dict(width=3, color='#dc2626'), marker=dict(size=10))
fig_d1_4.update_layout(height=420)
fig_d1_4

In [73]:
# Salvar Dashboard 1
save_dashboard(
    figs=[fig_d1_1, fig_d1_2, fig_d1_3, fig_d1_4],
    descriptions=[
        'Distribuição do delta entre entrega real e prazo estimado. Valores negativos indicam entrega adiantada; positivos, atraso.',
        'GRÁFICO-CHAVE DA HIPÓTESE 1: comparação direta da taxa de avaliação negativa entre pedidos entregues no prazo e pedidos atrasados.',
        'Box plot mostrando que pedidos com notas baixas (1-2) tendem a apresentar deltas positivos (atraso), enquanto notas altas (5) concentram-se em deltas negativos (adiantamento).',
        'Curva de severidade: a taxa de avaliação negativa cresce monotonicamente com o tamanho do atraso, confirmando a relação causal proposta.'
    ],
    title='Dashboard 1: Atrasos e Satisfação (Hipótese 1)',
    subtitle='Pedidos atrasados apresentam taxa significativamente maior de avaliações negativas',
    output_path=os.path.join(DASHBOARDS_PATH, 'dashboard_1_hipotese_1.html')
)

Dashboard salvo: /content/drive/MyDrive/PUCRS/projetobi-olist/dashboards/dashboard_1_hipotese_1.html


## 4. Dashboard 2: Hipótese 2 — Frete vs Preço em Baixo Ticket

**Hipótese 2:** em categorias de produtos de baixo ticket médio (inferior a R$ 50), o valor cobrado pelo frete exerce maior impacto negativo na satisfação do consumidor do que o preço nominal do produto.

In [74]:
# Box plot razao frete/preco por faixa
ordem_faixa = ['baixo', 'medio', 'alto']
df_razao = pd.read_sql('''
SELECT faixa_ticket, razao_frete_preco FROM fato_pedidos
WHERE razao_frete_preco BETWEEN 0 AND 3
''', engine)

fig_d2_1 = px.box(
    df_razao, x='faixa_ticket', y='razao_frete_preco',
    title='Razão frete/preço por faixa de ticket',
    category_orders={'faixa_ticket': ordem_faixa},
    color='faixa_ticket',
    labels={'faixa_ticket': 'Faixa de ticket', 'razao_frete_preco': 'Frete / Preço'}
)
fig_d2_1.update_layout(height=400, showlegend=False)
fig_d2_1

Output hidden; open in https://colab.research.google.com to view.

In [75]:
# PUNCH LINE da Hipotese 2: heatmap faixa x frete bucket x taxa_neg
heat = pd.read_sql('''
SELECT
    faixa_ticket,
    CASE
        WHEN razao_frete_preco < 0.1 THEN '1. <10%'
        WHEN razao_frete_preco < 0.25 THEN '2. 10-25%'
        WHEN razao_frete_preco < 0.5 THEN '3. 25-50%'
        WHEN razao_frete_preco < 1.0 THEN '4. 50-100%'
        ELSE '5. >100%'
    END AS faixa_frete,
    100.0 * AVG(avaliacao_negativa) AS taxa_neg_pct,
    COUNT(*) AS n
FROM fato_pedidos
GROUP BY faixa_ticket, faixa_frete
''', engine)

pivot = heat.pivot(index='faixa_frete', columns='faixa_ticket', values='taxa_neg_pct')
pivot = pivot[['baixo', 'medio', 'alto']]

fig_d2_2 = px.imshow(
    pivot.values, x=pivot.columns, y=pivot.index,
    color_continuous_scale='Reds', aspect='auto', text_auto='.1f',
    title='HIPÓTESE 2 - Taxa de avaliação negativa por faixa de ticket x peso do frete',
    labels=dict(x='Faixa de ticket', y='Razão frete/preço', color='Taxa neg (%)')
)
fig_d2_2.update_layout(height=450)
fig_d2_2

In [76]:
# Top categorias de baixo ticket com maior taxa de neg
cat_baixo = pd.read_sql('''
SELECT p.categoria,
       100.0 * AVG(f.avaliacao_negativa) AS taxa_neg_pct,
       COUNT(*) AS n_pedidos
FROM fato_pedidos f
JOIN dim_produto p ON f.produto_key = p.product_id
WHERE f.faixa_ticket = 'baixo' AND p.categoria IS NOT NULL
GROUP BY p.categoria
HAVING COUNT(*) >= 100
ORDER BY taxa_neg_pct DESC
LIMIT 10
''', engine)

fig_d2_3 = px.bar(
    cat_baixo.sort_values('taxa_neg_pct'),
    x='taxa_neg_pct', y='categoria', orientation='h', text='taxa_neg_pct',
    title='Top 10 categorias de baixo ticket com maior taxa de avaliação negativa',
    labels={'taxa_neg_pct': 'Taxa de aval. negativa (%)', 'categoria': 'Categoria'},
    color='taxa_neg_pct', color_continuous_scale='Reds'
)
fig_d2_3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_d2_3.update_layout(height=500, showlegend=False, coloraxis_showscale=False)
fig_d2_3

In [77]:
# Valor de frete por faixa de ticket
df_frete = pd.read_sql('''
SELECT faixa_ticket, valor_total_frete FROM fato_pedidos
WHERE valor_total_frete < 200
''', engine)

fig_d2_4 = px.box(
    df_frete, x='faixa_ticket', y='valor_total_frete',
    title='Distribuição do valor de frete por faixa de ticket',
    category_orders={'faixa_ticket': ordem_faixa},
    color='faixa_ticket',
    labels={'faixa_ticket': 'Faixa de ticket', 'valor_total_frete': 'Valor de frete (R$)'}
)
fig_d2_4.update_layout(height=400, showlegend=False)
fig_d2_4

In [78]:
# Salvar Dashboard 2
save_dashboard(
    figs=[fig_d2_1, fig_d2_2, fig_d2_3, fig_d2_4],
    descriptions=[
        'Em pedidos de baixo ticket, o frete frequentemente representa uma fração significativa (ou supera) o preço do produto.',
        'GRÁFICO-CHAVE DA HIPÓTESE 2: o canto superior esquerdo (ticket baixo + frete pesado) concentra as maiores taxas de avaliação negativa.',
        'Categorias de baixo ticket com maior insatisfação - candidatos prioritários para revisão da política de frete.',
        'Em termos absolutos, o valor de frete e similar entre faixas; o problema esta na razão frete/preço em ticket baixo.'
    ],
    title='Dashboard 2: Frete vs Preço em Baixo Ticket (Hipótese 2)',
    subtitle='Em ticket baixo, frete elevado e o principal vetor de insatisfacao',
    output_path=os.path.join(DASHBOARDS_PATH, 'dashboard_2_hipotese_2.html')
)

Dashboard salvo: /content/drive/MyDrive/PUCRS/projetobi-olist/dashboards/dashboard_2_hipotese_2.html


## 5. Dashboard 3: Hipótese 3 — Pedidos Multi-Vendedor

**Hipótese 3:** pedidos que contêm itens de múltiplos vendedores geram maior tempo de consolidação logística e, consequentemente, apresentam taxas mais elevadas de avaliações negativas.

In [79]:
# Distribuicao da contagem de vendedores por pedido
sellers = pd.read_sql('''
SELECT num_sellers_unicos, COUNT(*) AS n
FROM fato_pedidos
GROUP BY num_sellers_unicos
ORDER BY num_sellers_unicos
''', engine)

fig_d3_1 = px.bar(
    sellers.head(10), x='num_sellers_unicos', y='n', text='n',
    title='Distribuição de vendedores por pedido (escala log)',
    labels={'num_sellers_unicos': 'Vendedores únicos no pedido', 'n': 'Quantidade de pedidos'},
    log_y=True
)
fig_d3_1.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig_d3_1.update_layout(height=400)
fig_d3_1

In [80]:
# PUNCH LINE da Hipotese 3
h3_taxa = pd.read_sql('''
SELECT CASE WHEN multi_vendedor = 1 THEN 'Múltiplos vendedores' ELSE 'Único vendedor' END AS tipo,
       100.0 * AVG(avaliacao_negativa) AS taxa_neg_pct,
       COUNT(*) AS n_pedidos
FROM fato_pedidos GROUP BY multi_vendedor
''', engine)

fig_d3_2 = px.bar(
    h3_taxa, x='tipo', y='taxa_neg_pct', text='taxa_neg_pct',
    title='HIPÓTESE 3 - Taxa de avaliação negativa: único vs múltiplos vendedores',
    labels={'tipo': 'Tipo de pedido', 'taxa_neg_pct': 'Taxa de aval. negativa (%)'},
    color='tipo',
    color_discrete_map={'Único vendedor': '#16a34a', 'Múltiplos vendedores': '#dc2626'}
)
fig_d3_2.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_d3_2.update_layout(height=420, showlegend=False)
fig_d3_2

In [81]:
# Tempo de consolidacao single vs multi
df_consol = pd.read_sql('''
SELECT CASE WHEN multi_vendedor = 1 THEN 'Múltiplos vendedores' ELSE 'Único vendedor' END AS tipo,
       tempo_consolidacao_dias
FROM fato_pedidos
WHERE tempo_consolidacao_dias BETWEEN 0 AND 30
''', engine)

fig_d3_3 = px.box(
    df_consol, x='tipo', y='tempo_consolidacao_dias',
    title='Tempo de consolidação logística por tipo de pedido',
    color='tipo',
    color_discrete_map={'Único vendedor': '#16a34a', 'Múltiplos vendedores': '#dc2626'},
    labels={'tipo': 'Tipo', 'tempo_consolidacao_dias': 'Tempo de consolidacao (dias)'}
)
fig_d3_3.update_layout(height=400, showlegend=False)
fig_d3_3

In [82]:
# Proporcao de pedidos multi ao longo do tempo
multi_mensal = pd.read_sql('''
SELECT t.ano, t.mes,
       SUM(CASE WHEN f.multi_vendedor = 1 THEN 1 ELSE 0 END) AS multi,
       COUNT(*) AS total,
       100.0 * SUM(CASE WHEN f.multi_vendedor = 1 THEN 1 ELSE 0 END) / COUNT(*) AS pct_multi
FROM fato_pedidos f
JOIN dim_tempo t ON f.tempo_key = t.data_key
GROUP BY t.ano, t.mes
ORDER BY t.ano, t.mes
''', engine)
multi_mensal['data'] = pd.to_datetime(dict(year=multi_mensal['ano'], month=multi_mensal['mes'], day=1))

fig_d3_4 = px.line(
    multi_mensal, x='data', y='pct_multi', markers=True,
    title='Proporção de pedidos de múltiplos vendedores ao longo do tempo',
    labels={'data': 'Mês', 'pct_multi': '% de pedidos multi-vendedor'}
)
fig_d3_4.update_traces(line=dict(width=3, color='#7c3aed'), marker=dict(size=8))
fig_d3_4.update_layout(height=400)
fig_d3_4

In [83]:
# Salvar Dashboard 3
save_dashboard(
    figs=[fig_d3_1, fig_d3_2, fig_d3_3, fig_d3_4],
    descriptions=[
        'A grande maioria dos pedidos envolve um único vendedor; pedidos de múltiplos vendedores são minoria mas representam volume relevante (eixo Y em escala logarítmica).',
        'GRÁFICO-CHAVE DA HIPÓTESE 3: comparação direta da taxa de avaliação negativa entre pedidos único vendedor e múltiplos vendedores.',
        'Pedidos de ç apresentam tempo de consolidação logística visivelmente maior, sustentando o mecanismo causal da hipótese.',
        'Evolução temporal da participação de pedidos de múltiplos vendedores no volume total.'
    ],
    title='Dashboard 3: Pedidos de múltiplos vendedores (Hipótese 3)',
    subtitle='Pedidos com múltiplos vendedores exigem mais consolidação e sofrem mais avaliações negativas',
    output_path=os.path.join(DASHBOARDS_PATH, 'dashboard_3_hipotese_3.html')
)

Dashboard salvo: /content/drive/MyDrive/PUCRS/projetobi-olist/dashboards/dashboard_3_hipotese_3.html


## 6. Validação: dashboards gerados

Lista os arquivos HTML produzidos para conferência dentro da pasta `/dashboards/`.

In [84]:
arquivos = sorted(os.listdir(DASHBOARDS_PATH))
print(f'Dashboards em {DASHBOARDS_PATH}:\n')
for arq in arquivos:
    caminho = os.path.join(DASHBOARDS_PATH, arq)
    tamanho_kb = os.path.getsize(caminho) / 1024
    print(f'  {arq:50s} {tamanho_kb:>8.1f} KB')

Dashboards em /content/drive/MyDrive/PUCRS/projetobi-olist/dashboards:

  dashboard_0_visao_executiva.html                       38.2 KB
  dashboard_1_hipotese_1.html                          1258.0 KB
  dashboard_2_hipotese_2.html                          3839.5 KB
  dashboard_3_hipotese_3.html                          2072.6 KB
